# 01 · Define & Explore — TAA biology, CDR structure, epitope choice + the VHH hello-world

**Standard slot:** *define & explore.* **For Project 17 this means:** understand the tumor-associated
antigen (TAA) and the nanobody (VHH) you will design against it, **choose your epitope** (overlapping
vs non-overlapping with an approved mAb), fix the metrics table, and run the **mock** VHH hello-world
end-to-end (D0).

Run `00_setup.ipynb` first in this session. Everything here runs with **no GPU** on the deterministic
`mock` backend — switch to the real backend (RFantibody/BoltzGen) on Colab A100 in notebook 02.

## The biology in one screen

A **nanobody (VHH)** is the ~15 kDa single variable domain of a camelid heavy-chain-only antibody. It
folds as one immunoglobulin domain with three **CDR loops** (CDR1, CDR2, and the long, dominant
**CDR3**) presented on a stable framework. That small, stable, single-domain format is why nanobodies
power **tumor-imaging agents** (fast clearance, deep penetration) and **CAR / bispecific** binder
modules.

A **tumor-associated antigen (TAA)** is a cell-surface protein over-expressed on tumor cells. Classic
examples: **HER2** (ERBB2) and **EGFR** (ERBB1) in the HER/ErbB receptor family, and **mesothelin**.
De novo VHH design to a *defined, validated* TAA epitope is a real translational pipeline — but it is
hard, and hit rates are LOW, so we treat designs as **screening inputs**, not finished binders.

**The epitope choice is the central design decision (P0/P1).** Two strategies:
- **Overlapping** with an approved mAb's footprint (e.g., the trastuzumab epitope on HER2 domain IV)
  → you compete with / mimic a validated therapeutic site.
- **Non-overlapping / orthogonal** → enables **biparatopic** or **bispecific** constructs that bind
  alongside the mAb, and avoids resistance tied to the mAb site.

You will justify your choice with explicit, measurable success criteria in your D0 problem statement.

## The metrics table (what we will filter on)

| Metric | Range | Means | Does **not** mean | Cutoff (antibody) |
|--------|-------|-------|-------------------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the VHH model | thermostability / affinity | ≥ 70 |
| pae_interaction | Å | confidence in the VHH–antigen *interface* arrangement | binding/affinity | ≤ 12 |
| scRMSD | Å | designed-vs-predicted VHH backbone (self-consistency) | binding | ≤ 3.0 |
| CDR geometry RMSD | Å | CDR-loop geometry vs an IgFold/NanoBodyBuilder2 model | a good paratope | small (loop sanity) |
| TAP-like score | count | developability liability flags (proxy) | a real TAP call | fewer = better |
| CamSol-like | a.u. | solubility proxy (hydropathy) | a real CamSol score | higher = better |
| humanness | 0–1 | human-likeness proxy | low immunogenicity guarantee | higher = better |

The `"antibody"` cutoffs (scRMSD ≤ 3.0, pLDDT ≥ 70, pae_interaction ≤ 12) come from the shared
`filtering_pipeline.DEFAULT_CUTOFFS["antibody"]`. **Developability metrics here are TEACHING
HEURISTICS, not the validated tools (TAP/CamSol/Hu-mAb)** — see `MANUAL.md §2` and `antibody_tools.py`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Pick your TAA, epitope, and framework

These are the three fixed inputs to the campaign. The structures are **candidates — verify on RCSB in
Week 1** (`data/README.md`). The epitope residue lists below are **EXAMPLE placeholders** — derive the
real ones from the antigen surface (and, for the overlapping choice, from the mAb–antigen interface in
the reference complex).

In [ ]:
# --- Campaign definition (EDIT these in Week 1 after verifying the structures) ---
TAA = "HER2"                 # one of: HER2 (1N8Z), EGFR (1IVO), mesothelin (model) — verify on RCSB
REFERENCE_COMPLEX = "1N8Z"   # candidate: trastuzumab Fab - HER2 domain IV (verify; entries get superseded)

# EXAMPLE epitope residue tokens (chain+number). REPLACE with residues you read off the antigen
# surface. For the "overlapping" strategy, take residues from the mAb-antigen interface in 1N8Z;
# for "non-overlapping", choose a distinct patch (e.g., HER2 domain I/II away from the trastuzumab site).
EPITOPE_OVERLAPPING     = "A557,A560,A579,A580,A583"   # EXAMPLE — trastuzumab-region (verify!)
EPITOPE_NONOVERLAPPING  = "A245,A266,A270,A289"        # EXAMPLE — orthogonal patch (verify!)

# Choose one to drive the campaign; record WHY in your D0 problem statement.
EPITOPE = EPITOPE_OVERLAPPING
EPITOPE_STRATEGY = "overlapping-with-trastuzumab"   # or "non-overlapping-orthogonal"

print("TAA            =", TAA)
print("epitope        =", EPITOPE, "(", EPITOPE_STRATEGY, ")")
print("ref complex    =", REFERENCE_COMPLEX, "(candidate — verify on RCSB)")

## The VHH framework

The CDRs are the design variables; the **framework is fixed**. `antibody_tools.DEFAULT_FRAMEWORK` is a
humanized-VHH (huVHH3-style) **teaching placeholder** — verify and replace with the exact germline
framework you choose. Keeping a humanized framework helps the humanness axis from the start.

In [ ]:
from antibody_tools import DEFAULT_FRAMEWORK, parse_epitope, epitope_overlap

fw = DEFAULT_FRAMEWORK
print("framework:", fw["name"])
for k in ("FR1", "FR2", "FR3", "FR4"):
    print(f"  {k}: {fw[k]}")

ep = parse_epitope(EPITOPE)
print("\nparsed epitope residues:", ep)

## VHH hello-world (mock backend, no GPU)

Generate a few mock VHH designs against your epitope and assemble + score one. This proves the
plumbing (design → CDR loops → AF2-Multimer-ab metrics → developability) before you spend A100 time in
notebook 02. **Every number below is SYNTHETIC — never report mock numbers as real.**

In [ ]:
from antibody_tools import design_vhh_cdrs, score_designs, extract_cdrs

# Mock hello-world: 5 deterministic VHH candidates.
designs = design_vhh_cdrs(TAA, EPITOPE, framework=fw, n=5, tool="mock")
score_designs(designs, tool="mock")     # fills af2_multimer_ab + developability (SYNTHETIC)

d = designs[0]
print("design_id :", d.design_id)
print("VHH length:", len(d.sequence), "aa")
print("CDRs      :", extract_cdrs(d))
print("metrics   : pLDDT", d.plddt, "| pae_interaction", d.pae_interaction,
      "| scrmsd", d.scrmsd, "| cdr_geom", d.cdr_geom)
print("dev (heur):", "TAP-like", d.tap_score, "| CamSol-like", d.camsol_like,
      "| humanness", d.humanness)
print("epitope overlap:", epitope_overlap(d.contact_residues, EPITOPE))
print("synthetic :", d.synthetic, "->", d.notes)

## Visualize a structure (py3Dmol)

Use this to eyeball a real predicted VHH–antigen complex once you have one (notebook 02 on Colab). The
mock backend writes no PDB.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real RFantibody/AF2-Multimer run):
# show_pdb("results/af2/vhh_her2_complex.pdb")
print("show_pdb(pdb_path) ready — use it on a real VHH-antigen complex in notebook 02.")

## D0 checklist
- [ ] 1-page **problem statement**: the TAA, the chosen **epitope + strategy** (overlapping vs
      non-overlapping, with justification), and **measurable** success criteria.
- [ ] Verified `REFERENCE_COMPLEX` / TAA accessions on RCSB; epitope residues read off the real
      surface (not the EXAMPLE placeholders).
- [ ] Metric table understood, including the "does not mean" column and that developability numbers
      here are **heuristics**.
- [ ] Mock VHH hello-world run; CDR3 length + SYNTHETIC metrics printed.
- [ ] `LOG.md` entry (seed, what you ran).

**Next:** `02_generate.ipynb` — the VHH design campaign (mock now; RFantibody on A100).